# Expert-concept policies in a simulated MARL task

This notebook is a scoped reproduction of Zabounidis et al., [*Concept Learning for Interpretable Multi-Agent Reinforcement Learning*](https://proceedings.mlr.press/v205/zabounidis23a.html) (CoRL 2022 / PMLR 205, 2023), limited to one simulated cooperative-competitive task.

**Current evidence:** deterministic smoke-fixture training, held-out metrics, matched-control accounting, typed TensorDict intervention mechanics, and artifact provenance only. **The paper result is not evaluated.**

The PMLR paper (SHA-256 `7368cd383b7a32fc8efcd09eacbba1e6e6ca99eaf9b7638f18fdb9e8e491b015`) and supplement (SHA-256 `3654a69aa13262addb77121e5e0faf688f2324d61c8c78e5bdbc4450f05384ba`) specify the simulated FortAttack task, concepts, broad architecture, MAPPO budget, and evaluation count. They do not provide an immutable task implementation, author code, checkpoints, seed list, train/evaluation scenarios, hyperparameter-search protocol, or full reference outputs. No code or checkpoint is linked from the publication page. Scientific mode therefore fails closed until a licensed, digest-pinned bundle supplies those assets.

The bounded smoke path uses a notebook-local synthetic 2v2 lane-defense classification task. It is **not FortAttack**, uses supervised expert-action fitting rather than MAPPO, omits recurrence/IterNorm/continuous concepts, and reports expert-action agreement rather than episodic win rate. Passing it supports API and analysis mechanics only; it cannot support claims about the paper, FortAttack, MARL performance, sample efficiency, stability, or real robots.


## Reproduction contract and pinned deviations

| Item | Paper target | This notebook |
|---|---|---|
| Task | simulated FortAttack, declared here as the 2v2 defender setting | synthetic 2v2 lane defense smoke fixture |
| Concepts | Range, Strategy, Target, Orientation, Position | discrete Range, Strategy, Target only |
| Policy | shared recurrent MAPPO actor under CTDE | shared feed-forward expert-action classifier |
| Training | 10M timesteps, best checkpoint, Adam, scheduled LR/entropy, five plotted seeds/two evaluation seeds | five deterministic seeds, 80 full-batch steps, fixed Adam LR |
| Evaluation | 100 simulated episodes; win rate and concept errors | frozen 96-episode fixture; action agreement and concept-group accuracy |
| Intervention | oracle replacement of incorrect concepts during rollout | oracle, incorrect, shuffled, and no-op replacements on the frozen TensorDict |

Paper-pinned details retained in the admission manifest include: 2v2 hard concept width 13, soft concept width 9 with residual width 23, baseline residual width 128, 128-unit fully connected groups, LSTM maximum sequence length 50, IterNorm iterations `T=2`, concept-loss coefficient 10, batch size 10,240, minibatch size 1,600, Adam `(0.9, 0.999)`, and 100 simulated evaluation episodes. The paper does not identify sufficient immutable implementation detail to recreate those settings faithfully.

### Intervention and decision gates

- **Intervention:** overwrite the learned discrete concept-logit groups before the action head.
- **Controls:** no-op, deliberately incorrect group labels, and an episode-shuffled oracle; all use the same frozen evaluation TensorDict.
- **Metrics:** held-out Strategy/Target/Range accuracy, per-agent action agreement, episode-level all-agents-correct rate, seed stability, training-curve area, paired intervention effects, and episode bootstrap intervals. Reductions always name the `episode`, `agent`, or `concept_group` axes.
- **Scientific gate:** paper mode requires exact task/code/checkpoints/splits/reference outputs plus digest and license metadata. Smoke success never opens this gate.


In [1]:
import copy
import hashlib
import json
import os
import subprocess

import torch
from tensordict import TensorDict
from tensordict.nn import TensorDictModule
from tdhook.latent import SteeringVectors
from tdhook.workflow import Workflow
from xdrl import (
    ArtifactDigestAlgorithm,
    BatchSemantics,
    InputArtifactReference,
    InputArtifactRole,
    InteractionContract,
    InteractionPhase,
    KeyPresence,
    KeyRole,
    KeySchema,
    ModelRole,
    OutputArtifactDeclaration,
    OutputArtifactDigest,
    OutputArtifactRole,
    RuntimeInteractionContext,
    TDHookWorkflowRunner,
    TensorDictSchema,
)

MODE = os.environ.get("XDRL_MARL_CONCEPT_MODE", "smoke")
SEED = 5900
TRAIN_EPISODES = 256
EVALUATION_EPISODES = 96
AGENTS = 2
OBSERVATION_DIM = 12
HIDDEN_DIM = 16
CONCEPT_DIM = 7
ACTION_ORDER = ("TURN_LEFT", "TURN_RIGHT", "ADVANCE", "WAIT", "TAG")
CONCEPT_GROUPS = {"strategy": slice(0, 3), "target": slice(3, 5), "range": slice(5, 7)}
TRAINING_SEEDS = (5900, 5901, 5902, 5903, 5904)
TRAINING_STEPS = 80
LEARNING_RATE = 0.03
CONCEPT_LOSS_COEFFICIENT = 1.0
PAPER_URL = "https://proceedings.mlr.press/v205/zabounidis23a.html"
PAPER_PDF_SHA256 = "7368cd383b7a32fc8efcd09eacbba1e6e6ca99eaf9b7638f18fdb9e8e491b015"
SUPPLEMENT_PDF_SHA256 = "3654a69aa13262addb77121e5e0faf688f2324d61c8c78e5bdbc4450f05384ba"
REFERENCE_ASSET_BUNDLE = None


def repository_revision():
    revision = subprocess.run(["git", "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
    dirty = subprocess.run(
        ["git", "status", "--porcelain", "--untracked-files=no"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    return f"{revision}+dirty" if dirty else revision


CODE_REVISION = repository_revision()
if MODE not in {"smoke", "paper"}:
    raise ValueError("XDRL_MARL_CONCEPT_MODE must be 'smoke' or 'paper'")
if MODE == "paper" and REFERENCE_ASSET_BUNDLE is None:
    raise RuntimeError(
        "paper mode is blocked: a licensed, immutable FortAttack implementation, author checkpoints, frozen splits, "
        "seed/config manifests, and full reference outputs with digests are required"
    )

paper_contract = {
    "task": "FortAttack simulated 2v2 defenders",
    "task_implementation_revision": None,
    "author_code": None,
    "author_checkpoints": None,
    "reference_outputs": None,
    "concept_definitions": {
        "range": "per opponent: within 0.8 map units and pi/5 radians; binary one-hot",
        "strategy": "team-level left, right, or random; categorical one-hot",
        "target": "selected opposing agent; categorical one-hot",
        "orientation": "relative angle to each opponent; continuous",
        "position": "relative Euclidean distance to each opponent; continuous",
    },
    "architecture_2v2": {
        "hard_concept_width": 13,
        "soft_concept_width": 9,
        "soft_residual_width": 23,
        "baseline_residual_width": 128,
        "recurrent": "LSTM",
        "iterative_normalization_steps": 2,
    },
    "training": {
        "algorithm": "MAPPO",
        "timesteps": 10_000_000,
        "batch_size": 10_240,
        "minibatch_size": 1_600,
        "optimizer": "Adam",
        "betas": (0.9, 0.999),
        "concept_loss_coefficient": 10,
    },
    "evaluation": {"simulated_episodes": 100, "trained_policy_seeds_reported": 2},
    "underspecified_reimplementation_choices": [
        "exact FortAttack source revision and license",
        "complete observations/actions/rewards and termination implementation",
        "oracle source code and group ordering",
        "initialization, train/evaluation scenarios, and seed identities",
        "hyperparameter-search space and checkpoint-selection trace",
        "IterNorm implementation details and full MAPPO configuration",
    ],
}
paper_contract

{'task': 'FortAttack simulated 2v2 defenders',
 'task_implementation_revision': None,
 'author_code': None,
 'author_checkpoints': None,
 'reference_outputs': None,
 'concept_definitions': {'range': 'per opponent: within 0.8 map units and pi/5 radians; binary one-hot',
  'strategy': 'team-level left, right, or random; categorical one-hot',
  'target': 'selected opposing agent; categorical one-hot',
  'orientation': 'relative angle to each opponent; continuous',
  'position': 'relative Euclidean distance to each opponent; continuous'},
 'architecture_2v2': {'hard_concept_width': 13,
  'soft_concept_width': 9,
  'soft_residual_width': 23,
  'baseline_residual_width': 128,
  'recurrent': 'LSTM',
  'iterative_normalization_steps': 2},
 'training': {'algorithm': 'MAPPO',
  'timesteps': 10000000,
  'batch_size': 10240,
  'minibatch_size': 1600,
  'optimizer': 'Adam',
  'betas': (0.9, 0.999),
  'concept_loss_coefficient': 10},
 'evaluation': {'simulated_episodes': 100, 'trained_policy_seeds_r

## Generate a frozen, axis-explicit smoke fixture

Each episode has two defenders (`agent` axis). The hidden opponent strategy is shared at episode level; target and range labels are agent-specific. The observation contains noisy encodings of these labels plus nuisance features. The expert action is a declared deterministic function of the three concept groups. Train and evaluation generators and episode IDs are disjoint.


In [2]:
def make_fixture(episodes, *, seed, noise):
    generator = torch.Generator().manual_seed(seed)
    strategy = torch.randint(3, (episodes, 1), generator=generator).expand(-1, AGENTS).clone()
    target = torch.randint(2, (episodes, AGENTS), generator=generator)
    in_range = torch.randint(2, (episodes, AGENTS), generator=generator)
    strategy_one_hot = torch.nn.functional.one_hot(strategy, 3).float()
    target_one_hot = torch.nn.functional.one_hot(target, 2).float()
    range_one_hot = torch.nn.functional.one_hot(in_range, 2).float()
    concepts = torch.cat((strategy_one_hot, target_one_hot, range_one_hot), dim=-1)

    mixing = torch.tensor(
        [
            [1.0, -0.4, 0.2, 0.7, -0.2, 0.4, -0.6],
            [-0.3, 0.9, 0.1, -0.6, 0.8, -0.5, 0.3],
            [0.2, -0.5, 1.0, 0.4, 0.1, 0.7, -0.3],
            [0.8, 0.2, -0.4, 0.5, -0.7, 0.2, 0.6],
            [-0.6, 0.1, 0.7, -0.2, 0.6, 0.8, -0.4],
            [0.4, 0.6, -0.3, 0.9, 0.2, -0.6, 0.1],
            [0.1, -0.7, 0.5, 0.3, 0.9, -0.1, 0.4],
            [0.5, 0.3, 0.6, -0.8, 0.1, 0.2, 0.7],
        ]
    )
    signal = concepts @ mixing.T
    nuisance = torch.randn(episodes, AGENTS, 4, generator=generator)
    observation = torch.cat((signal + noise * torch.randn(signal.shape, generator=generator), nuisance), dim=-1)
    action = torch.where(in_range.bool(), torch.full_like(strategy, 4), (strategy + 2 * target) % 4)
    return {
        "episode_id": torch.arange(seed * 10_000, seed * 10_000 + episodes),
        "observation": observation,
        "concept": concepts,
        "strategy": strategy,
        "target": target,
        "range": in_range,
        "action": action,
    }


train_fixture = make_fixture(TRAIN_EPISODES, seed=SEED + 10, noise=0.28)
evaluation_fixture = make_fixture(EVALUATION_EPISODES, seed=SEED + 20, noise=0.62)
assert set(train_fixture["episode_id"].tolist()).isdisjoint(evaluation_fixture["episode_id"].tolist())
assert evaluation_fixture["observation"].shape == (EVALUATION_EPISODES, AGENTS, OBSERVATION_DIM)
{
    "mode": MODE,
    "task": "synthetic-2v2-lane-defense-smoke-v1",
    "ordered_batch_axes": ("episode", "agent"),
    "concept_groups": {name: (group.start, group.stop) for name, group in CONCEPT_GROUPS.items()},
    "action_order": ACTION_ORDER,
    "train_episode_digest": hashlib.sha256(train_fixture["episode_id"].numpy().tobytes()).hexdigest(),
    "evaluation_episode_digest": hashlib.sha256(evaluation_fixture["episode_id"].numpy().tobytes()).hexdigest(),
    "scientific_claim_ready": False,
}

{'mode': 'smoke',
 'task': 'synthetic-2v2-lane-defense-smoke-v1',
 'ordered_batch_axes': ('episode', 'agent'),
 'concept_groups': {'strategy': (0, 3), 'target': (3, 5), 'range': (5, 7)},
 'action_order': ('TURN_LEFT', 'TURN_RIGHT', 'ADVANCE', 'WAIT', 'TAG'),
 'train_episode_digest': '242c681ed0e3dbecad5fdf5c863546b03d4a9f8e7f48e0564e54b0d270aadd30',
 'evaluation_episode_digest': '50a64e59c3ac8459760888247dce84c3e0ebbd15f8aaf01f87a4c6497aab778c',
 'scientific_claim_ready': False}

## Train concept and matched non-concept policies

Both arms have the same architecture, initialization, parameter count, examples, optimizer, learning rate, steps, and action objective. The only intended difference is the auxiliary grouped concept loss. To match parameters exactly, the smoke non-concept arm retains the same grouped bottleneck but receives no concept supervision; it is not the paper's width-128 residual baseline. This smoke architecture is deliberately smaller than—and not a substitute for—the paper architecture.


In [3]:
class BottleneckPolicy(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = torch.nn.Sequential(torch.nn.Linear(OBSERVATION_DIM, HIDDEN_DIM), torch.nn.Tanh())
        self.concept_head = torch.nn.Linear(HIDDEN_DIM, CONCEPT_DIM)
        self.action_head = torch.nn.Linear(CONCEPT_DIM, len(ACTION_ORDER))

    def forward(self, observation):
        concept_logits = self.concept_head(self.encoder(observation))
        concept_probabilities = torch.cat(
            [torch.softmax(concept_logits[..., group], dim=-1) for group in CONCEPT_GROUPS.values()], dim=-1
        )
        return concept_logits, self.action_head(concept_probabilities)


def grouped_concept_loss(logits, fixture):
    losses = []
    for name, group in CONCEPT_GROUPS.items():
        losses.append(torch.nn.functional.cross_entropy(logits[..., group].flatten(0, 1), fixture[name].flatten()))
    return torch.stack(losses).mean()


def action_accuracy(logits, target):
    return float((logits.argmax(-1) == target).float().mean())


def train_pair(seed):
    torch.manual_seed(seed)
    initial = BottleneckPolicy()
    concept_model = copy.deepcopy(initial)
    baseline_model = copy.deepcopy(initial)
    concept_optimizer = torch.optim.Adam(concept_model.parameters(), lr=LEARNING_RATE)
    baseline_optimizer = torch.optim.Adam(baseline_model.parameters(), lr=LEARNING_RATE)
    curves = {"concept": [], "baseline": []}
    for _step in range(TRAINING_STEPS):
        for name, model, optimizer, auxiliary in (
            ("concept", concept_model, concept_optimizer, True),
            ("baseline", baseline_model, baseline_optimizer, False),
        ):
            optimizer.zero_grad()
            concept_logits, action_logits = model(train_fixture["observation"])
            loss = torch.nn.functional.cross_entropy(action_logits.flatten(0, 1), train_fixture["action"].flatten())
            if auxiliary:
                loss = loss + CONCEPT_LOSS_COEFFICIENT * grouped_concept_loss(concept_logits, train_fixture)
            loss.backward()
            optimizer.step()
            with torch.no_grad():
                eval_action_logits = model(evaluation_fixture["observation"])[1]
                curves[name].append(action_accuracy(eval_action_logits, evaluation_fixture["action"]))
    return concept_model.eval(), baseline_model.eval(), curves


runs = {seed: train_pair(seed) for seed in TRAINING_SEEDS}
selected_model, selected_baseline, _ = runs[SEED]
parameter_counts = {
    "concept": sum(parameter.numel() for parameter in selected_model.parameters()),
    "baseline": sum(parameter.numel() for parameter in selected_baseline.parameters()),
}
assert parameter_counts["concept"] == parameter_counts["baseline"]
training_budget = {
    "parameter_counts": parameter_counts,
    "examples_per_step": TRAIN_EPISODES * AGENTS,
    "optimizer": "Adam",
    "learning_rate": LEARNING_RATE,
    "steps": TRAINING_STEPS,
    "seeds": TRAINING_SEEDS,
    "matched": True,
    "declared_difference": "grouped auxiliary concept loss only",
}
training_budget

{'parameter_counts': {'concept': 367, 'baseline': 367},
 'examples_per_step': 512,
 'optimizer': 'Adam',
 'learning_rate': 0.03,
 'steps': 80,
 'seeds': (5900, 5901, 5902, 5903, 5904),
 'matched': True,
 'declared_difference': 'grouped auxiliary concept loss only'}

## Held-out concept, policy, stability, and sample-efficiency summaries

Concept accuracy is computed separately for each group over the named `(episode, agent)` axes. Policy accuracy is first reduced over agents and episodes; the team proxy requires both agents' actions to match the fixture expert. Training-curve area and first threshold crossing are descriptive smoke summaries, not environment sample-efficiency claims.


In [4]:
def evaluate_model(model):
    with torch.inference_mode():
        concepts, actions = model(evaluation_fixture["observation"])
    group_accuracy = {
        name: float((concepts[..., group].argmax(-1) == evaluation_fixture[name]).float().mean(dim=(0, 1)))
        for name, group in CONCEPT_GROUPS.items()
    }
    correct_by_agent = actions.argmax(-1) == evaluation_fixture["action"]
    return {
        "concept_accuracy_reduction": "mean over (episode, agent), reported per concept_group",
        "concept_accuracy": group_accuracy,
        "policy_accuracy_reduction": "mean over (episode, agent)",
        "agent_action_accuracy": float(correct_by_agent.float().mean(dim=(0, 1))),
        "per_agent_action_accuracy": correct_by_agent.float().mean(dim=0).tolist(),
        "episode_all_agents_correct": float(correct_by_agent.all(dim=1).float().mean(dim=0)),
    }


per_seed = {}
for seed, (concept_model, baseline_model, curves) in runs.items():
    per_seed[seed] = {
        "concept": evaluate_model(concept_model),
        "baseline": evaluate_model(baseline_model),
        "training_curve": {
            arm: {
                "area_mean": float(torch.tensor(values).mean()),
                "first_step_at_0.80": next((index + 1 for index, value in enumerate(values) if value >= 0.80), None),
                "final": values[-1],
            }
            for arm, values in curves.items()
        },
    }


def seed_summary(arm, metric):
    values = torch.tensor([per_seed[seed][arm][metric] for seed in TRAINING_SEEDS])
    return {"mean": float(values.mean()), "sample_std": float(values.std()), "values": values.tolist()}


held_out_summary = {
    "selected_seed": SEED,
    "selected_concept_policy": per_seed[SEED]["concept"],
    "selected_nonconcept_policy": per_seed[SEED]["baseline"],
    "stability_across_training_seeds": {
        arm: {
            "agent_action_accuracy": seed_summary(arm, "agent_action_accuracy"),
            "episode_all_agents_correct": seed_summary(arm, "episode_all_agents_correct"),
        }
        for arm in ("concept", "baseline")
    },
    "descriptive_training_curve_summaries": {seed: per_seed[seed]["training_curve"] for seed in TRAINING_SEEDS},
}
held_out_summary

{'selected_seed': 5900,
 'selected_concept_policy': {'concept_accuracy_reduction': 'mean over (episode, agent), reported per concept_group',
  'concept_accuracy': {'strategy': 0.8072916865348816,
   'target': 0.875,
   'range': 0.9479166865348816},
  'policy_accuracy_reduction': 'mean over (episode, agent)',
  'agent_action_accuracy': 0.8385416865348816,
  'per_agent_action_accuracy': [0.8229166865348816, 0.8541666865348816],
  'episode_all_agents_correct': 0.7083333134651184},
 'selected_nonconcept_policy': {'concept_accuracy_reduction': 'mean over (episode, agent), reported per concept_group',
  'concept_accuracy': {'strategy': 0.53125,
   'target': 0.5416666865348816,
   'range': 0.8541666865348816},
  'policy_accuracy_reduction': 'mean over (episode, agent)',
  'agent_action_accuracy': 0.6979166865348816,
  'per_agent_action_accuracy': [0.65625, 0.7395833134651184],
  'episode_all_agents_correct': 0.4895833432674408},
 'stability_across_training_seeds': {'concept': {'agent_action_a

## Adapt the selected concept policy to XDRL's public TensorDict boundary

The TensorDict batch has ordered `episode` and `agent` axes. The contract keeps concept features and action classes in separate output keys; no reduction conflates the agent, concept-group, or action-class spaces. Native PyTorch and typed XDRL execution must agree exactly before intervention.


In [5]:
policy_core = copy.deepcopy(selected_model).eval()
policy = TensorDictModule(policy_core, in_keys=["observation"], out_keys=["concept_logits", "action_logits"])
evaluation_batch = TensorDict(
    {"observation": evaluation_fixture["observation"].clone()},
    batch_size=[EVALUATION_EPISODES, AGENTS],
)
batch_semantics = BatchSemantics(("episode", "agent"))
contract = InteractionContract(
    identity="marl-concept-policy:synthetic-2v2:evaluation",
    role=ModelRole.ACTOR,
    phase=InteractionPhase.EVALUATION,
    module_path="policy",
    input_schema=TensorDictSchema(
        (KeySchema("observation", KeyRole.OBSERVATION, KeyPresence.REQUIRED),), batch_semantics
    ),
    output_schema=TensorDictSchema(
        (
            KeySchema("concept_logits", KeyRole.FEATURE, KeyPresence.PRODUCED),
            KeySchema("action_logits", KeyRole.ACTION, KeyPresence.PRODUCED),
        ),
        batch_semantics,
    ),
    model_id="synthetic-concept-bottleneck-smoke",
    checkpoint_id=f"synthetic:seed-{SEED}:step-{TRAINING_STEPS}",
    module_training=False,
)
interaction = RuntimeInteractionContext(contract, policy, evaluation_batch)
with torch.inference_mode():
    native_concepts, native_actions = policy_core(evaluation_fixture["observation"])
    adapted = interaction(evaluation_batch.clone())
adapter_parity = torch.equal(native_concepts, adapted["concept_logits"]) and torch.equal(
    native_actions, adapted["action_logits"]
)
assert adapter_parity
{
    "batch_axes": batch_semantics.dimensions,
    "concept_tensor_shape": tuple(adapted["concept_logits"].shape),
    "action_tensor_shape": tuple(adapted["action_logits"].shape),
    "native_xdrl_output_parity": adapter_parity,
}

{'batch_axes': ('episode', 'agent'),
 'concept_tensor_shape': (96, 2, 7),
 'action_tensor_shape': (96, 2, 5),
 'native_xdrl_output_parity': True}

## Frozen concept interventions and provenance

Every arm hooks the public `module.concept_head` target through TDHook's `SteeringVectors` workflow and runs through `TDHookWorkflowRunner.run_paired` for each of the five independently trained models. Correct replacement uses the frozen oracle groups; incorrect replacement rotates every group label; shuffled replacement permutes whole episodes while retaining each episode's agent axis. The baseline arm is an explicit no-op. All seeds share the same frozen evaluation TensorDict, while checkpoint and full-result artifacts remain seed-specific. XDRL's pair manifest establishes matched mechanics and artifact identity—not causality.


In [6]:
def tensor_digest(tensor):
    contiguous = tensor.detach().cpu().contiguous()
    payload = (
        str(contiguous.dtype).encode() + json.dumps(list(contiguous.shape)).encode() + contiguous.numpy().tobytes()
    )
    return hashlib.sha256(payload).hexdigest()


def module_digest(module):
    digest = hashlib.sha256()
    for name, value in sorted(module.state_dict().items()):
        digest.update(name.encode())
        digest.update(tensor_digest(value).encode())
    return digest.hexdigest()


def named_tensor_digest(items):
    digest = hashlib.sha256()
    for name, value in items:
        digest.update(name.encode())
        digest.update(tensor_digest(value).encode())
    return digest.hexdigest()


def logits_from_fixture(fixture, *, wrong=False):
    parts = []
    for name, width in (("strategy", 3), ("target", 2), ("range", 2)):
        labels = fixture[name]
        if wrong:
            labels = (labels + 1) % width
        parts.append(16.0 * torch.nn.functional.one_hot(labels, width).float() - 8.0)
    return torch.cat(parts, dim=-1)


oracle_logits = logits_from_fixture(evaluation_fixture)
incorrect_logits = logits_from_fixture(evaluation_fixture, wrong=True)
shuffle_generator = torch.Generator().manual_seed(SEED + 30)
episode_permutation = torch.randperm(EVALUATION_EPISODES, generator=shuffle_generator)
shuffled_logits = oracle_logits[episode_permutation]


def keep_concepts(*, output, **_):
    return output


def replacement_callback(label, replacement):
    def replace(*, output, **_):
        assert replacement.shape == output.shape
        return replacement.to(device=output.device, dtype=output.dtype)

    replace.__name__ = f"replace_{label}_concepts"
    return replace


callbacks = {
    "correct": replacement_callback("correct", oracle_logits),
    "incorrect": replacement_callback("incorrect", incorrect_logits),
    "shuffled": replacement_callback("shuffled", shuffled_logits),
}
frozen_items = (
    ("episode_id", evaluation_fixture["episode_id"]),
    ("observation", evaluation_fixture["observation"]),
    ("concept", evaluation_fixture["concept"]),
    ("action", evaluation_fixture["action"]),
)
frozen_evaluation_digest = named_tensor_digest(frozen_items)
shared_inputs = (
    InputArtifactReference(
        "evaluation:synthetic-2v2-frozen-v1",
        InputArtifactRole.EVALUATION_SPLIT,
        ArtifactDigestAlgorithm.SHA256,
        frozen_evaluation_digest,
        metadata={
            "frozen": True,
            "ordered_batch_axes": ["episode", "agent"],
            "episodes": EVALUATION_EPISODES,
            "agents": AGENTS,
            "components": [name for name, _ in frozen_items],
        },
    ),
    InputArtifactReference(
        "reference:zabounidis-et-al-2023-paper",
        InputArtifactRole.OTHER,
        ArtifactDigestAlgorithm.SHA256,
        PAPER_PDF_SHA256,
        source="https://proceedings.mlr.press/v205/zabounidis23a/zabounidis23a.pdf",
        revision="PMLR-v205",
        source_is_immutable=True,
        metadata={"document_role": "paper", "reference_results_available": False},
    ),
    InputArtifactReference(
        "reference:zabounidis-et-al-2023-supplement",
        InputArtifactRole.OTHER,
        ArtifactDigestAlgorithm.SHA256,
        SUPPLEMENT_PDF_SHA256,
        source="https://proceedings.mlr.press/v205/zabounidis23a/zabounidis23a-supp.pdf",
        revision="PMLR-v205",
        source_is_immutable=True,
        metadata={"document_role": "supplement", "reference_results_available": False},
    ),
)


def result_resolver(data, declarations):
    digest = named_tensor_digest(
        (("concept_logits", data["concept_logits"]), ("action_logits", data["action_logits"]))
    )
    return tuple(OutputArtifactDigest(item.identity, ArtifactDigestAlgorithm.SHA256, digest) for item in declarations)


def intervention_context(training_seed):
    core = copy.deepcopy(runs[training_seed][0]).eval()
    seed_policy = TensorDictModule(core, in_keys=["observation"], out_keys=["concept_logits", "action_logits"])
    seed_contract = InteractionContract(
        identity=f"marl-concept-policy:synthetic-2v2:seed-{training_seed}:evaluation",
        role=ModelRole.ACTOR,
        phase=InteractionPhase.EVALUATION,
        module_path="policy",
        input_schema=TensorDictSchema(
            (KeySchema("observation", KeyRole.OBSERVATION, KeyPresence.REQUIRED),), batch_semantics
        ),
        output_schema=TensorDictSchema(
            (
                KeySchema("concept_logits", KeyRole.FEATURE, KeyPresence.PRODUCED),
                KeySchema("action_logits", KeyRole.ACTION, KeyPresence.PRODUCED),
            ),
            batch_semantics,
        ),
        model_id="synthetic-concept-bottleneck-smoke",
        checkpoint_id=f"synthetic:seed-{training_seed}:step-{TRAINING_STEPS}",
        module_training=False,
    )
    checkpoint = InputArtifactReference(
        f"checkpoint:synthetic-concept-policy-seed-{training_seed}",
        InputArtifactRole.MODEL_CHECKPOINT,
        ArtifactDigestAlgorithm.SHA256,
        module_digest(core),
        metadata={
            "mode": MODE,
            "paper_checkpoint": False,
            "training_seed": training_seed,
            "training_steps": TRAINING_STEPS,
        },
    )
    interaction = RuntimeInteractionContext(seed_contract, seed_policy, evaluation_batch)
    return {
        "core": core,
        "runner": TDHookWorkflowRunner(interaction),
        "input_artifacts": (checkpoint,) + shared_inputs,
    }


seed_contexts = {training_seed: intervention_context(training_seed) for training_seed in TRAINING_SEEDS}


def run_intervention_pair(training_seed, label, callback):
    context = seed_contexts[training_seed]
    pair = context["runner"].run_paired(
        Workflow(SteeringVectors(["module.concept_head"], steer_fn=keep_concepts)),
        Workflow(SteeringVectors(["module.concept_head"], steer_fn=callback)),
        evaluation_batch,
        pair_id=f"marl-concept-policy:smoke:seed-{training_seed}:{label}",
        code_revision=CODE_REVISION,
        declared_workflow_differences=(0,),
        seed=training_seed,
        input_artifacts=context["input_artifacts"],
        baseline_output_artifacts=(
            OutputArtifactDeclaration(
                f"result:seed-{training_seed}:{label}:no-op", OutputArtifactRole.INTERVENTION_RESULT
            ),
        ),
        baseline_output_artifact_resolver=result_resolver,
        intervention_output_artifacts=(
            OutputArtifactDeclaration(
                f"result:seed-{training_seed}:{label}:intervention", OutputArtifactRole.INTERVENTION_RESULT
            ),
        ),
        intervention_output_artifact_resolver=result_resolver,
        callback_identifiers={keep_concepts: "no-op", callback: f"replace-{label}-concept-groups"},
    )
    assert pair.manifest.interpretation == "mechanics_and_provenance_only"
    assert pair.baseline.provenance.input_artifacts == pair.intervention.provenance.input_artifacts
    return pair


pairs_by_seed = {
    training_seed: {
        label: run_intervention_pair(training_seed, label, callback) for label, callback in callbacks.items()
    }
    for training_seed in TRAINING_SEEDS
}
{
    "workflow_target": "module.concept_head",
    "training_seeds": TRAINING_SEEDS,
    "frozen_evaluation_sha256": frozen_evaluation_digest,
    "checkpoint_sha256": {
        training_seed: seed_contexts[training_seed]["input_artifacts"][0].digest_value
        for training_seed in TRAINING_SEEDS
    },
    "result_artifact_sha256": {
        training_seed: {
            label: pair.intervention.provenance.output_artifacts[0].digest_value for label, pair in seed_pairs.items()
        }
        for training_seed, seed_pairs in pairs_by_seed.items()
    },
}

{'workflow_target': 'module.concept_head',
 'training_seeds': (5900, 5901, 5902, 5903, 5904),
 'frozen_evaluation_sha256': '3a78e8aeb84bd63856581fc6e20c687144a8b4d1c71863c66ac8c84e7bc964d4',
 'checkpoint_sha256': {5900: '72394cc4649bd3c909d6c82721344bc32860ab77ca41c449bf96d4cab8dc8111',
  5901: '1a7edaddfe1b2c6591e16fbce550cf650afd4e538aba358129f9b2399e0da73d',
  5902: 'c46cf9a67d40a296499b088bebe55958700880a194496a8f89863e6127cc068f',
  5903: '9efb9c44e9c5a4937d62941e0db374a7bd2d4d3fa3faa80cbddd205d676a0e58',
  5904: 'ee8b4a321ed52c9666c46b67b43e245700e97b5171bf3eaada4c88afb984d603'},
 'result_artifact_sha256': {5900: {'correct': '12b5d15afc945c0689a0717933931d426666686e6fc9f793d8528ae2d1a86bdc',
   'incorrect': 'a6798ed0e044da9250438395e99d10ea6b6d2b3a2128a767eff599340aaa0eb8',
   'shuffled': 'a155811cbf6736d90dc7988fea672ed9e63db40cb30c94c19322d571d2c7fa49'},
  5901: {'correct': '2c4181c1cb2dd68d606afbeb0c7b967335d217a6c7f457c264d8d32f6fa214e4',
   'incorrect': 'afb4626ab2898a958e9a

## Directional effects and uncertainty

Effects are paired on episode ID within each independently trained model. Agent action agreement is reduced over `(episode, agent)`; the episode metric is reduced over `agent` with logical `all`, then averaged over `episode`. The primary interval reduces each model to one paired mean effect, then bootstraps those five training-seed means (`n=5`). Within-seed episode bootstrap intervals are retained and explicitly labeled as conditional on one trained model. This separates evaluation-episode uncertainty from training-seed uncertainty. A correction is directionally supportive in this smoke fixture only when the seed-bootstrap lower bound is non-negative and the incorrect/shuffled controls do not show a larger mean gain. This gate is diagnostic and cannot admit a paper claim.


In [7]:
def paired_metrics(action_logits):
    correct = action_logits.argmax(-1) == evaluation_fixture["action"]
    return {
        "agent_correct_by_episode": correct.float().mean(dim=1),
        "all_agents_correct_by_episode": correct.all(dim=1).float(),
        "action_by_episode_agent": action_logits.argmax(-1),
    }


def bootstrap_mean_interval(values, *, seed, draws):
    generator = torch.Generator().manual_seed(seed)
    estimates = []
    for _ in range(draws):
        indices = torch.randint(len(values), (len(values),), generator=generator)
        estimates.append(values[indices].mean())
    return [float(value) for value in torch.stack(estimates).quantile(torch.tensor((0.025, 0.975)))]


def summarize_seed_pair(training_seed, label, pair):
    baseline = paired_metrics(pair.baseline.data["action_logits"])
    changed = paired_metrics(pair.intervention.data["action_logits"])
    agent_delta = changed["agent_correct_by_episode"] - baseline["agent_correct_by_episode"]
    team_delta = changed["all_agents_correct_by_episode"] - baseline["all_agents_correct_by_episode"]
    bootstrap_offset = {"correct": 0, "incorrect": 1, "shuffled": 2}[label]
    return {
        "training_seed": training_seed,
        "n_evaluation_episodes": EVALUATION_EPISODES,
        "agent_action_accuracy": {
            "baseline": float(baseline["agent_correct_by_episode"].mean()),
            "intervention": float(changed["agent_correct_by_episode"].mean()),
            "paired_delta": float(agent_delta.mean()),
            "within_seed_episode_bootstrap_95pct": bootstrap_mean_interval(
                agent_delta, seed=training_seed + 100 + bootstrap_offset, draws=500
            ),
        },
        "episode_all_agents_correct": {
            "baseline": float(baseline["all_agents_correct_by_episode"].mean()),
            "intervention": float(changed["all_agents_correct_by_episode"].mean()),
            "paired_delta": float(team_delta.mean()),
            "within_seed_episode_bootstrap_95pct": bootstrap_mean_interval(
                team_delta, seed=training_seed + 200 + bootstrap_offset, draws=500
            ),
        },
        "behavior_change_rate": float(
            (changed["action_by_episode_agent"] != baseline["action_by_episode_agent"]).float().mean(dim=(0, 1))
        ),
    }


per_seed_intervention_results = {
    training_seed: {label: summarize_seed_pair(training_seed, label, pair) for label, pair in seed_pairs.items()}
    for training_seed, seed_pairs in pairs_by_seed.items()
}


def aggregate_metric(label, metric, *, seed):
    baseline = torch.tensor(
        [per_seed_intervention_results[training_seed][label][metric]["baseline"] for training_seed in TRAINING_SEEDS]
    )
    intervention = torch.tensor(
        [
            per_seed_intervention_results[training_seed][label][metric]["intervention"]
            for training_seed in TRAINING_SEEDS
        ]
    )
    paired_delta = intervention - baseline
    return {
        "n_training_seeds": len(TRAINING_SEEDS),
        "training_seed_ids": list(TRAINING_SEEDS),
        "evaluation_episodes_per_seed": EVALUATION_EPISODES,
        "baseline_mean_across_seeds": float(baseline.mean()),
        "intervention_mean_across_seeds": float(intervention.mean()),
        "paired_delta_by_seed": paired_delta.tolist(),
        "mean_paired_delta_across_seeds": float(paired_delta.mean()),
        "training_seed_bootstrap_95pct": bootstrap_mean_interval(paired_delta, seed=seed, draws=5000),
    }


intervention_results = {
    label: {
        "reduction_contract": {
            "within_seed": "paired mean over frozen episodes; agent axis retained within episode",
            "across_seed": "one paired mean per independently trained model; bootstrap over training_seed",
        },
        "agent_action_accuracy": aggregate_metric(label, "agent_action_accuracy", seed=SEED + 300 + index),
        "episode_all_agents_correct": aggregate_metric(label, "episode_all_agents_correct", seed=SEED + 400 + index),
        "per_seed": {
            training_seed: per_seed_intervention_results[training_seed][label] for training_seed in TRAINING_SEEDS
        },
    }
    for index, label in enumerate(callbacks)
}
correct_lower = intervention_results["correct"]["agent_action_accuracy"]["training_seed_bootstrap_95pct"][0]
correct_gain = intervention_results["correct"]["agent_action_accuracy"]["mean_paired_delta_across_seeds"]
largest_control_gain = max(
    intervention_results[label]["agent_action_accuracy"]["mean_paired_delta_across_seeds"]
    for label in ("incorrect", "shuffled")
)
directional_smoke_gate = {
    "uncertainty_unit": "training_seed",
    "n_training_seeds": len(TRAINING_SEEDS),
    "correct_seed_bootstrap_lower_bound_nonnegative": correct_lower >= 0.0,
    "correct_mean_gain_exceeds_controls": correct_gain > largest_control_gain,
    "passed": correct_lower >= 0.0 and correct_gain > largest_control_gain,
    "scope": "synthetic smoke fixture only",
}
{"effects": intervention_results, "directional_smoke_gate": directional_smoke_gate}

{'effects': {'correct': {'reduction_contract': {'within_seed': 'paired mean over frozen episodes; agent axis retained within episode',
    'across_seed': 'one paired mean per independently trained model; bootstrap over training_seed'},
   'agent_action_accuracy': {'n_training_seeds': 5,
    'training_seed_ids': [5900, 5901, 5902, 5903, 5904],
    'evaluation_episodes_per_seed': 96,
    'baseline_mean_across_seeds': 0.8041666746139526,
    'intervention_mean_across_seeds': 0.8979166746139526,
    'paired_delta_by_seed': [0.08854162693023682,
     0.078125,
     0.1302083134651184,
     0.08854162693023682,
     0.08333331346511841],
    'mean_paired_delta_across_seeds': 0.09374997764825821,
    'training_seed_bootstrap_95pct': [0.08229164779186249,
     0.11249997466802597]},
   'episode_all_agents_correct': {'n_training_seeds': 5,
    'training_seed_ids': [5900, 5901, 5902, 5903, 5904],
    'evaluation_episodes_per_seed': 96,
    'baseline_mean_across_seeds': 0.6625000238418579,
    'i

## Final evidence ledger

Execution, asset availability, descriptive metrics, intervention mechanics, directional smoke behavior, reference agreement, and scientific claim readiness remain separate. Negative controls and failed gates are retained. A future scientific run must replace the smoke fixture—not reinterpret it—with a digest-pinned FortAttack implementation, author-compatible training/evaluation assets, declared seeds, exact oracle/group ordering, checkpoint-selection records, and reference outputs.


In [8]:
verdict = {
    "smoke_execution": "passed",
    "paper_exact_assets": "unavailable",
    "paper_mode": "blocked",
    "task_implementation": "notebook-local synthetic 2v2 lane-defense fixture",
    "training_method": "supervised expert-action fitting; not MAPPO",
    "held_out_metrics": held_out_summary,
    "matched_training_budget": training_budget,
    "intervention_effects": intervention_results,
    "directional_smoke_gate": directional_smoke_gate,
    "matched_provenance": {
        training_seed: {label: pair.manifest.to_dict() for label, pair in seed_pairs.items()}
        for training_seed, seed_pairs in pairs_by_seed.items()
    },
    "reference_agreement": "not evaluated",
    "fortattack_policy_performance": "not evaluated",
    "scientific_claim_ready": False,
    "claim_limit": (
        "No inference to FortAttack, the paper's checkpoints or results, other MARL tasks, real robots, "
        "or concept-policy sample efficiency and stability broadly."
    ),
}
{
    "smoke_execution": verdict["smoke_execution"],
    "paper_mode": verdict["paper_mode"],
    "intervention_training_seeds": list(TRAINING_SEEDS),
    "matched_intervention_pairs": sum(len(seed_pairs) for seed_pairs in pairs_by_seed.values()),
    "directional_smoke_gate": verdict["directional_smoke_gate"],
    "reference_agreement": verdict["reference_agreement"],
    "scientific_claim_ready": verdict["scientific_claim_ready"],
    "claim_limit": verdict["claim_limit"],
}

{'smoke_execution': 'passed',
 'paper_mode': 'blocked',
 'intervention_training_seeds': [5900, 5901, 5902, 5903, 5904],
 'matched_intervention_pairs': 15,
 'directional_smoke_gate': {'uncertainty_unit': 'training_seed',
  'n_training_seeds': 5,
  'correct_seed_bootstrap_lower_bound_nonnegative': True,
  'correct_mean_gain_exceeds_controls': True,
  'passed': True,
  'scope': 'synthetic smoke fixture only'},
 'reference_agreement': 'not evaluated',
 'scientific_claim_ready': False,
 'claim_limit': "No inference to FortAttack, the paper's checkpoints or results, other MARL tasks, real robots, or concept-policy sample efficiency and stability broadly."}